# EDA - Marathon

### Schema and nr of columns
- Nr of columns: 13

In [0]:
VOLUME_PATH = '/Volumes/marathos/default/raw'

RAWDATA_PATH = VOLUME_PATH + '/data'
COUNTRIESDATA_PATH = VOLUME_PATH + '/countries'

RAWDATA_FILE = RAWDATA_PATH + '/TWO_CENTURIES_OF_UM_RACES.csv'
COUNTRIESDATA_FILE = COUNTRIESDATA_PATH + '/code_countries.csv'

auto_schema = (
    spark.read.format("csv")
    .options(header=True, inferSchema=True)
    .load(f"{RAWDATA_FILE}")
    .schema
)

print(f"Nr of columns: {len(auto_schema.fields)}")
print(auto_schema)

In [0]:
auto_schema = (
    spark.read.format("csv")
    .options(header=True, inferSchema=True)
    .load(f"{COUNTRIESDATA_FILE}")
    .schema
)

print(f"Nr of columns: {len(auto_schema.fields)}")
print(auto_schema)

df_countries = (
    spark.read.format("csv")
    .options(header=True, inferSchema=True)
    .load(f"{COUNTRIESDATA_FILE}")
)

df_countries.createOrReplaceTempView("df_countries_view")

df_countries.limit(10).display()

#### Structure new schema and create a view
- New schema to have the option to change the schema if necessary.
- Create a dataframe and a temporal view to be able to examine the data.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

real_schema = StructType([
    StructField('Year of event', IntegerType(), True),
    StructField('Event dates', StringType(), True),
    StructField('Event name', StringType(), True),
    StructField('Event distance/length', StringType(), True),
    StructField('Event number of finishers', IntegerType(),True),
    StructField('Athlete performance', StringType(), True),
    StructField('Athlete club', StringType(), True),
    StructField('Athlete country', StringType(), True),
    StructField('Athlete year of birth', DoubleType(), True),
    StructField('Athlete gender', StringType(), True),
    StructField('Athlete age category', StringType(), True),
    StructField('Athlete average speed', StringType(), True),
    StructField('Athlete ID', IntegerType(), True)])

df_marathon = (
    spark.read.format("csv")
    .options(header=True, schema=real_schema)
    .load(f"{RAWDATA_FILE}")
)

df_marathon.createOrReplaceTempView("df_marathon_view")

df_marathon.limit(10).display()

### Counts all Nulls for all columns
Nulls values
- Athlete performance: 2
- Athlete club: 2 826 373
- Athlete country: 3
- Athlete year of birth: 588 161
- Athlete gender: 7
- Athlete age category: 584 938
- Athlete average speed: 224

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_marathon.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df_marathon.columns]
)

null_counts = null_counts.collect()[0].asDict()

[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

### Counts all rows
- Nr of rows: 7 461 195

In [0]:
%sql
SELECT
    count(*) AS nr_rows
FROM df_marathon_view;

### Which countries are most represented?


In [0]:
from pyspark.sql.functions import col

df_top_countries = df_marathon.filter(col("Athlete country").isNotNull()) \
                              .groupBy("Athlete country").count() \
                              .orderBy(col("count").desc()) \
                              .limit(10)

df_top_countries.plot.bar(x="Athlete country", y="count")

#### Examine these columns to see how they relate to each other
- Event distance/length
- Athlete performance
- Athlete average speed

If the Event is a distance event (mi or km) then performance will be in time (h). If it is a time event then performance will be in distance (km). Regardless, Athlete average speed will always be in speed which means I will be able to compare runners on speed regardless of whether they ran a distance event or a time event.

#### how many unique events are their?
- 26907

In [0]:
%sql
SELECT
    COUNT(DISTINCT `Event name`) AS nr_events
FROM marathos.bronze.raw_marathon;

In [0]:
%sql
SELECT
    `Athlete age category` AS Category,
    count(*)
FROM df_marathon_view
GROUP BY `Athlete age category`
ORDER BY `Athlete age category`;


In [0]:
%sql
SELECT
    `Event distance/length` AS event_goal,
    `Athlete performance` AS performance,
    `Athlete average speed` AS avg_speed,
    CASE
        WHEN `Event distance/length` LIKE "%km%" THEN "km"
        WHEN `Event distance/length` LIKE "%mi%" THEN "km"
        WHEN `Event distance/length` LIKE "%h%" THEN "h"
    END AS Unit
FROM df_marathon_view
WHERE `Event distance/length` LIKE "%mi%"
LIMIT 40;

#### Examine Athlete average speed
Sometimes the Athlete average speed is in time HH:mm:ss and it seam to be the time it took to finish the race. To correct this information we could do a count between distance and time.

In [0]:
%sql
SELECT
    `Event name` AS Event,
    `Event distance/length` AS time_length,
    `Athlete performance` AS preformance,
    `Athlete average speed` AS avg_speed,
    TRY_CAST(`Athlete average speed` AS FLOAT) AS avg_speed_int
FROM df_marathon_view
WHERE TRY_CAST(`Athlete average speed` AS FLOAT) IS NULL
  AND `Athlete average speed` IS NOT NULL
GROUP BY
    `Athlete average speed`,
    `Event name`,
    `Event distance/length`,
    `Athlete performance`
ORDER BY `Athlete average speed` DESC;

#### Examine gender
Investigating gender as if there are more categories than male and female.

It turns out that in addition to F and M, there is also X and null value. Since null is when the value is missing, X should be when you have left a response, but have not answered M or F.



In [0]:
%sql
SELECT
    count_if(`Athlete gender`="M") AS male,
    count_if(`Athlete gender`="F") AS female,
    count_if(`Athlete gender`="X") AS X,
    count_if(`Athlete gender`=NULL) AS nr_null
FROM df_marathon_view
LIMIT 40;

#### Examine age category
Investigating gender as if there are more categories than male and female.

Analytic
- 37 categories
- W or F = Female
- M = Male

If ther is a U in the class it is for contesters under the age according to https://northstowehalf.co.uk/age-groups/  
- MU20 = Male under 20
- WU23 = Female under 23


In [0]:
%sql
SELECT
    `Athlete age category` AS Category,
    count(*)
FROM df_marathon_view
GROUP BY `Athlete age category`
ORDER BY `Athlete age category`;

### How are ages distributed among the runners?

In [0]:
from pyspark.sql.functions import col

df_age = df_marathon.withColumn(
    "Athlete_Age", 
    (col("Year of event").cast("double") - col("Athlete year of birth").cast("double")).cast("int")
)

df_grouped = df_age.filter(col("Athlete_Age").isNotNull() & (col("Athlete_Age") <= 100)) \
                   .groupBy("Athlete_Age").count() \
                   .orderBy("Athlete_Age")

df_grouped.plot.bar(x="Athlete_Age", y="count")

In [0]:
%sql
SELECT
    `Athlete age category` AS Category,
    count(*)
FROM df_marathon_view
GROUP BY `Athlete age category`
ORDER BY `Athlete age category`;

## Initial Analys Summary

- Rows: 7 461 195
- Columns: 13

| Column | Shows |Datatype now|Datatype should be|Nr nulls|
|--------|-------|------------|-------------------|--------|
|Year of event|A year (YYYY) for the event|Integer|Integer|0|
|Event dates|The date for the event|String|Date|0|
|Event name|The event name and country code for the event<br>26907 unique events|String|String|0|
|Event distance/length|The distance (km/mi) and nr etapps or the runingtime (h) for the event|String|String|0|
|Event number of finishers|How many compleated the event|Integer|Integer|0|
|Athlete performance|Either the distance run in a specific time or the time it took to run a specific distance|String|String|2|
|Athlete club|The Club that the competitor runs for|String|String|2826373|
|Athlete country|The country the competitor is running for|String|String|3|
|Athlete year of birth|The year the contestant was born|Double|Double|588161|
|Athlete gender|M=male <br> F=female <br> X=Not specified <br> null=Missing info|String|String|7|
|Athlete age category|37 categories<br>W or F = Female <br>M = Male<br>MU and WU = Under the age Female resp Male|String|String|584938|
|Athlete average speed|average speed of contestant|String|Float|224|
|Athlete ID|ID for specific contestant|Integer|Integer|0|